# 05 — Robustness, Time Stability and the NDE Comparison: Audit

MallWorld audit (2026-10)

---

## Scope

This notebook re-derives:
- *Ontological Versus Cultural Structure in MallWorld* (archived notebook 11): location type × atmosphere and vertical position × atmosphere, with time-split, author-holdout, drop-top-k and within-author permutation checks;
- the thesis's temporal claims (§7.1–7.2, §8.5);
- the MallWorld–NDE comparison (archived notebook 10; thesis abstract "V = 0.645"; thesis §7.3 "V = 0.352").

**Pre-stated reading.** None of these checks can separate a cultural explanation from a correspondential one.
- Robustness to authors and time shows that a pattern is shared and stable.
- A shared cultural schema predicts the same.
- The archived report says this itself (§4.2), and its title promises more than its tests can deliver.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

L = mw.load_locations()
A = L[(L["atmosphere"] != "not_mentioned")].reset_index(drop=True)
def cramers_v_bc(tab):
    """Bias-corrected Cramér's V (Bergsma 2013)."""
    tab = np.asarray(tab, float); n = tab.sum(); r, k = tab.shape
    chi2 = chi2_contingency(tab, correction=False)[0]; phi2 = chi2 / n
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1)); rc = r - (r - 1) ** 2 / (n - 1); kc = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / min(kc - 1, rc - 1))

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


## R1 — Location type × atmosphere

**Old.** χ² = 1,737.99, df = 621, V = 0.212 (4,306 locations); within-author null V = 0.059, observed 0.170 (p = 0.001).

In [2]:
tab = pd.crosstab(A["location_type"], A["atmosphere"])
chi, p, dof, _ = chi2_contingency(tab)
print(f"Full table {tab.shape}: n = {int(tab.values.sum())}, chi2 = {chi:.2f}, df = {dof}, V = {mw.cramers_v(tab):.3f}, bias-corrected V = {cramers_v_bc(tab):.3f}; {mw.expected_count_report(tab)}")
dreams_per_author = posts[posts["primary"]].groupby("author")["post_id"].nunique()
single = A["author"].map(dreams_per_author).eq(1)
print(f"Locations from authors with a single dream: {100 * single.mean():.1f}% -> for these, 'within-author' shuffling is within-dream shuffling")

top = A["location_type"].value_counts(); keep = top[top >= 20].index
At = A[A["location_type"].isin(keep)].reset_index(drop=True)
def v_of(df):
    return mw.cramers_v(pd.crosstab(df["location_type"], df["atmosphere"]))
v_obs = v_of(At)
rng = np.random.default_rng(5)
def shuffled(df, by):
    out = df.copy(); out["atmosphere"] = out.groupby(by)["atmosphere"].transform(lambda s: rng.permutation(s.values)); return out
null_auth = np.array([v_of(shuffled(At, "author")) for _ in range(300)])
null_dream = np.array([v_of(shuffled(At, "post_id")) for _ in range(300)])
null_all = np.array([v_of(At.assign(atmosphere=rng.permutation(At["atmosphere"].values))) for _ in range(300)])
print(f"Types with >= 20 coded locations: {len(keep)}; n = {len(At)}; observed V = {v_obs:.3f}")
for nm, nl in [("no structure (global shuffle)", null_all), ("within-author shuffle", null_auth), ("within-dream shuffle", null_dream)]:
    print(f"  null {nm:30s}: mean {nl.mean():.3f}, 99th pct {np.percentile(nl, 99):.3f}")

Full table (70, 10): n = 4293, chi2 = 1729.52, df = 621, V = 0.212, bias-corrected V = 0.170; min expected = 0.03; cells with expected < 5: 74%
Locations from authors with a single dream: 53.6% -> for these, 'within-author' shuffling is within-dream shuffling


Types with >= 20 coded locations: 41; n = 3985; observed V = 0.195
  null no structure (global shuffle) : mean 0.100, 99th pct 0.108
  null within-author shuffle         : mean 0.121, 99th pct 0.133
  null within-dream shuffle          : mean 0.129, 99th pct 0.138


### Findings — R1

**Reproduced.** χ² = 1,729.52, df = 621, V = 0.212 over 4,293 primary locations; the archived value was 1,737.99 over 4,306.

**But 74% of the cells have an expected count below 5.** A V on a sparse 70 × 10 table is inflated by chance. With the 41 types that have at least 20 coded locations:

| | V |
|---|---|
| Observed | 0.195 |
| Labels shuffled at random (no structure at all) | 0.100 |
| Shuffled within author | 0.121 |
| Shuffled within dream | 0.129 |
| Bias-corrected V on the full table | 0.170 |

**Verdict.** Location type carries atmosphere information beyond each dream's overall tone: observed 0.195 against a within-dream null with a 99th percentile of 0.138. The archived conclusion that the association is not an author artifact stands.

Two qualifications:
- **"Within-author" is mostly "within-dream".** 53.6% of locations come from authors with a single dream, and for them within-author shuffling is within-dream shuffling.
- **The null was not "less than half" the observed value.** The within-author null is 62% of the observed V. The archived null (0.059) does not reproduce.

None of these checks bears on "ontological vs cultural". The clustered, length-adjusted estimate of the location-type effect is in notebook 02, C9.

## R2 — Vertical position × atmosphere

**Old.** χ² = 203.50, df = 45, V = 0.144 (1,955 locations). The corrected analysis of this association is notebook 02, C1.

In [3]:
Av = A[A["vertical"] != "not_mentioned"]
tab = pd.crosstab(Av["vertical"], Av["atmosphere"])
chi, p, dof, _ = chi2_contingency(tab)
print(f"Vertical x atmosphere {tab.shape}: n = {int(tab.values.sum())}, chi2 = {chi:.2f}, df = {dof}, V = {mw.cramers_v(tab):.3f}, bias-corrected V = {cramers_v_bc(tab):.3f}; {mw.expected_count_report(tab)}")
print("Rows:", list(tab.index))

Vertical x atmosphere (6, 10): n = 1952, chi2 = 201.82, df = 45, V = 0.144, bias-corrected V = 0.127; min expected = 2.78; cells with expected < 5: 10%
Rows: ['ground', 'lower', 'lowest', 'upper', 'uppermost', 'varies']


### Findings — R2

**Reproduced:** χ² = 201.82, df = 45, V = 0.144 (bias-corrected 0.127), over 1,952 locations.
- The table includes a "varies" row, and it treats six vertical levels as unordered categories.
- That says nothing about the predicted direction.

The directional, clustered analysis is in notebook 02, C1:
- underground worse than ground (**partial hit**);
- no improvement above ground level (**miss**).

## R3 — Time stability

**Old.**
- Report: "V = 0.169 vs 0.178, spread p = 0.54: no drift".
- Thesis §7.1: "all Cochran's Q p > 0.50"; §8.5: "instability 2021–2023".

A non-significant difference is not evidence of stability (CLAUDE.md §4.11). Here the difference between periods is estimated with a dream-level bootstrap CI and compared with an equivalence margin fixed in advance: |ΔV| < 0.05, and for odds ratios a ratio of ORs between 0.67 and 1.5.

In [4]:
P = posts[posts["primary"]].copy()
cut = P["date"].median()
At["recent"] = At["post_id"].map(P.set_index("post_id")["date"] >= cut)
print(f"Split at the median post date {str(cut)[:10]}: earlier {int((~At['recent']).sum())} / recent {int(At['recent'].sum())} locations")
ids_e = At.loc[~At["recent"], "post_id"].unique(); ids_r = At.loc[At["recent"], "post_id"].unique()
g = {k: d for k, d in At.groupby("post_id")}
def boot_v(ids):
    pick = rng.choice(ids, len(ids)); return v_of(pd.concat([g[i] for i in pick], ignore_index=True))
v_e, v_r = v_of(At[~At["recent"]]), v_of(At[At["recent"]])
diffs = np.array([boot_v(ids_r) - boot_v(ids_e) for _ in range(300)])
lo90, hi90 = np.percentile(diffs, [5, 95])
print(f"V earlier {v_e:.3f}, recent {v_r:.3f}; difference {v_r - v_e:+.3f}, 90% CI [{lo90:+.3f}, {hi90:+.3f}] -> equivalent within 0.05: {(lo90 > -0.05) and (hi90 < 0.05)}")
print("(Raw V depends on n per cell; both halves are of similar size.)")

L2 = L.dropna(subset=["valence"]).copy()
L2["neg"] = (L2["valence"] == -1).astype(float)
L2["year"] = L2["post_id"].map(P.set_index("post_id")["date"].dt.year)
print("\nNegative atmosphere by year (share of coded locations):")
for y, d in L2.groupby("year"):
    p_, lo, hi, n_, C = (lambda s: (s.mean(), *[np.nan] * 2, len(s), d["post_id"].nunique()))(d["neg"])
    print(f"  {int(y)}: {100 * p_:.1f}% (n = {n_}, dreams = {C})")
Lv = L2.dropna(subset=["vertical_level"]).copy()
Lv["under"] = (Lv["vertical_level"] < 0).astype(int); Lv["elev"] = (Lv["vertical_level"] > 0).astype(int)
Lv["recent"] = Lv["post_id"].map(P.set_index("post_id")["date"] >= cut).astype(int)
r = mw.clustered_logit(Lv, "neg ~ (under + elev) * recent + log_words")
print("\nVertical effect by period (interaction terms = ratio of ORs, recent vs earlier):")
print(mw.odds_ratio_table(r, ["under", "elev", "under:recent", "elev:recent"]).round(3).to_string())

Split at the median post date 2025-01-03: earlier 2152 / recent 1833 locations


V earlier 0.214, recent 0.229; difference +0.015, 90% CI [-0.002, +0.040] -> equivalent within 0.05: True
(Raw V depends on n per cell; both halves are of similar size.)

Negative atmosphere by year (share of coded locations):
  2021: 60.4% (n = 96, dreams = 21)
  2022: 79.4% (n = 247, dreams = 59)
  2023: 59.4% (n = 751, dreams = 227)
  2024: 69.3% (n = 1132, dreams = 370)
  2025: 65.7% (n = 1762, dreams = 617)
  2026: 66.9% (n = 157, dreams = 53)



Vertical effect by period (interaction terms = ratio of ORs, recent vs earlier):
                 OR     lo     hi      p
under         2.558  1.560  4.195  0.000
elev          1.243  0.873  1.768  0.227
under:recent  0.794  0.401  1.570  0.507
elev:recent   1.192  0.711  1.998  0.505


### Findings — R3

- **Location-type association.** This is stable across the median date split (2025-01-03): V = 0.214 earlier and 0.229 recent, difference +0.015 (90% CI −0.002 to +0.040). **Equivalent** within the pre-set margin of ±0.05.
- **Underground penalty.**
  - It holds in both periods: underground vs ground OR 2.56 (1.56–4.20) earlier.
  - The ratio of ORs, recent against earlier, is 0.79 (0.40–1.57). The CI is too wide to show equivalence or a change: **underdetermined**.
- **Yearly negative-atmosphere shares** range from 59.4% (2023) to 79.4% (2022).
  - The early years rest on 21 dreams (2021) and 59 dreams (2022).
  - "Instability 2021–2023" is the small samples of those years, not a finding.
- **No source.** "All Cochran's Q p > 0.50" (thesis §7.1) appears in no notebook output.

## R4 — MallWorld vs NDE atmosphere

**Old.** χ² = 4,739.51, df = 2, V = 0.645. The archived comparison had several problems:
- It set 5,279 MallWorld **locations** from the thesis population against 6,753 NDE **accounts**, two of them exact duplicates.
- NDE `mixed` was merged into neutral.
- Thesis §7.3 gives V = 0.352 for the same comparison, a figure that appears in no output.

In [5]:
# Both projects have a top-level `models` package; load the NDE one in isolation, then restore MallWorld's
NDE_ROOT = MW_ROOT.parent / "nde"
saved = {k: sys.modules.pop(k) for k in list(sys.modules) if k == "models" or k.startswith("models.")}
sys.path[:0] = [str(NDE_ROOT / "scripts"), str(NDE_ROOT)]
import nde_dataset as nd
nde = nd.load_frame()
for k in [k for k in sys.modules if k == "models" or k.startswith("models.")]:
    del sys.modules[k]
sys.modules.update(saved)
del sys.path[:2]
NDE3 = {"incredibly_pleasant": "positive", "pleasant": "positive", "mixed": "neutral/mixed", "neutral": "neutral/mixed",
        "distressing": "negative", "terrifying": "negative"}
assert set(NDE3) | {"not_mentioned"} >= set(nde["overall_valence"].unique())
nv = nde["overall_valence"].map(NDE3).dropna()
print(f"NDE accounts with a stated valence: {len(nv):,} of {len(nde):,}:", (100 * nv.value_counts(normalize=True)).round(1).to_dict())
print(f"  of which 'mixed': {int((nde['overall_valence'] == 'mixed').sum()):,}")

Lx = mw.load_locations("extractable")
mwl = Lx["valence"].dropna().map({-1: "negative", 0: "neutral/mixed", 1: "positive"})
tab = pd.DataFrame({"MallWorld locations": mwl.value_counts(), "NDE accounts": nv.value_counts()}).fillna(0)
chi, p, dof, _ = chi2_contingency(tab)
print(f"Archived construction (thesis-population locations vs NDE accounts): chi2 = {chi:.1f}, df = {dof}, V = {mw.cramers_v(tab):.3f}")

# Dream as the unit: a dream is negative if most of its coded locations are negative, positive if most are positive
Lp = L.dropna(subset=["valence"])
dm = Lp.groupby("post_id")["valence"].mean()
dlab = pd.cut(dm, [-1.01, -0.5, 0.5, 1.01], labels=["negative", "neutral/mixed", "positive"], right=False).astype(str)
print(f"MallWorld dreams with a coded atmosphere: {len(dlab):,}:", (100 * dlab.value_counts(normalize=True)).round(1).to_dict())
tab2 = pd.DataFrame({"MallWorld dreams": dlab.value_counts(), "NDE accounts": nv.value_counts()}).fillna(0)
chi2_, p2_, dof2_, _ = chi2_contingency(tab2)
print(f"Dream vs account: chi2 = {chi2_:.1f}, df = {dof2_}, V = {mw.cramers_v(tab2):.3f}")

NDE accounts with a stated valence: 6,123 of 6,751: {'neutral/mixed': 48.5, 'positive': 47.9, 'negative': 3.7}
  of which 'mixed': 2,788
Archived construction (thesis-population locations vs NDE accounts): chi2 = 5007.0, df = 2, V = 0.668
MallWorld dreams with a coded atmosphere: 1,347: {'negative': 61.8, 'neutral/mixed': 28.1, 'positive': 10.1}
Dream vs account: chi2 = 3127.8, df = 2, V = 0.647


### Findings — R4

**Archived construction.** On the verified loaders (NDE: 6,751 unique accounts), the archived construction gives V = 0.668.

**Matched units.** With units closer to each other, the difference remains: MallWorld dreams against NDE accounts gives V = 0.647.

| | Negative | Neutral or mixed | Positive |
|---|---|---|---|
| MallWorld dreams (n=1,347; majority of coded locations) | 61.8% | 28.1% | 10.1% |
| NDE accounts (n=6,123 with a stated valence) | 3.7% | 48.5% | 47.9% |

Of the 2,968 NDE accounts in the middle category, 2,788 are "mixed". Mixed is not neutral, so an NDE is rarely *neutral*.

**What this shows and what it does not.**
- The two archives differ greatly in atmosphere. The thesis's alternative figure, V = 0.352 (§7.3), appears in no output.
- The comparison does **not** discriminate between explanations:
  - The framework's reading, that MallWorld samples lower or intermediate states and NDEs higher ones, predicts the difference.
  - So does selection. r/themallworld collects uncanny mall dreams; NDE archives collect near-death experiences, which are predominantly pleasant by reputation and by self-selection.
  - The two archives were also coded with different schemas and prompts.
- **Status:** descriptive. Not a test of the framework.